# 연속 캔들 패턴 연구

**핵심 질문들:**
1. N번 음봉 연속 후 다음 봉이 양봉일 확률은?
2. 연속 음봉이 길어질수록 반등 확률이 높아지는가?
3. 연속 음봉 후 매수 전략은 실제로 수익이 나는가?
4. 몇 번 음봉 후 매수가 가장 좋은가? (파라미터 스윕)
5. 이 패턴이 우연인가, 통계적으로 유의한가?
6. 강세장/약세장에서 다르게 작동하는가?


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import yfinance as yf

from ginlix_backtest.engine import portfolio, signals, sweep
from ginlix_backtest.engine import rolling as rolling_bt
from ginlix_backtest.analysis import regime as regime_mod, stats as stat_mod, tearsheet

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. 데이터 로드

In [ ]:
SYMBOL = 'SPY'
START  = '2000-01-01'
END    = '2024-12-31'

raw = yf.download(SYMBOL, start=START, end=END, auto_adjust=True, progress=False)
close = raw['Close'].squeeze()
close.index = pd.to_datetime(close.index).tz_localize('UTC')

ret = close.pct_change().dropna()
benchmark = close.copy()

print(f'{SYMBOL}: {close.index[0].date()} ~ {close.index[-1].date()}, {len(close):,} bars')

## 2. 기초 분석: 연속 음봉/양봉 분포

먼저 실제로 연속 음봉이 얼마나 자주, 얼마나 길게 이어지는지 파악한다.

In [ ]:
streak = signals.candle_streak(close)

# 연속 음봉 구간 (streak < 0)
red_streaks = streak[streak < 0].abs()
# 연속 양봉 구간
green_streaks = streak[streak > 0]

print('=== 연속 음봉 통계 ===')
print(red_streaks.value_counts().sort_index().to_string())

print('\n=== 연속 양봉 통계 ===')
print(green_streaks.value_counts().sort_index().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

red_vc = red_streaks.value_counts().sort_index()
green_vc = green_streaks.value_counts().sort_index()

axes[0].bar(red_vc.index, red_vc.values, color='#e74c3c', edgecolor='white')
axes[0].set_title(f'{SYMBOL} - 연속 음봉 발생 횟수 분포')
axes[0].set_xlabel('연속 음봉 일수')
axes[0].set_ylabel('발생 횟수')
for x, y in zip(red_vc.index, red_vc.values):
    axes[0].text(x, y + 1, str(y), ha='center', fontsize=8)

axes[1].bar(green_vc.index, green_vc.values, color='#2ecc71', edgecolor='white')
axes[1].set_title(f'{SYMBOL} - 연속 양봉 발생 횟수 분포')
axes[1].set_xlabel('연속 양봉 일수')
axes[1].set_ylabel('발생 횟수')

plt.suptitle(f'연속 캔들 분포 ({START}~{END})', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## 3. 핵심: N번 음봉 후 다음 봉이 양봉일 확률

이것이 가장 중요한 분석이다.  
**완전 랜덤이라면** 음봉/양봉 확률은 약 50%로 일정해야 한다.  
음봉이 길어질수록 **평균 회귀** 효과가 있다면 확률이 올라가야 한다.

In [ ]:
def prob_after_n_reds(close, max_n=8):
    """N번 연속 음봉 후 다음 봉의 방향 확률 분석."""
    ret = close.pct_change()
    is_red   = (ret < 0).astype(int)
    is_green = (ret > 0).astype(int)
    
    rows = []
    for n in range(1, max_n + 1):
        # n번 연속 음봉인 날
        streak_n = is_red.rolling(n).sum() == n
        
        # 그 다음 날의 방향
        next_green = is_green.shift(-1)  # 다음 봉이 양봉?
        next_ret   = ret.shift(-1)        # 다음 봉의 수익률
        
        subset_green = next_green[streak_n].dropna()
        subset_ret   = next_ret[streak_n].dropna()
        
        n_occur = len(subset_green)
        if n_occur == 0:
            continue
            
        p_green = subset_green.mean()
        avg_ret = subset_ret.mean() * 100
        std_ret = subset_ret.std() * 100
        
        rows.append({
            'n_reds': n,
            'occurrences': n_occur,
            'p_next_green': p_green,
            'avg_next_ret_%': avg_ret,
            'std_next_ret_%': std_ret,
            'expected_value': avg_ret,  # 기댓값 = 평균 수익률
        })
    return pd.DataFrame(rows)

prob_df = prob_after_n_reds(close, max_n=9)

# 베이스라인: 아무 조건 없을 때 양봉 확률
baseline_p = (close.pct_change() > 0).mean()
print(f'베이스라인 양봉 확률 (조건 없음): {baseline_p:.1%}')
print()
display(prob_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 양봉 확률
axes[0].bar(prob_df['n_reds'], prob_df['p_next_green'] * 100,
            color=['#e74c3c' if v < baseline_p*100 else '#2ecc71' for v in prob_df['p_next_green']*100],
            edgecolor='white')
axes[0].axhline(baseline_p * 100, color='black', linestyle='--', linewidth=1.2,
                label=f'baseline {baseline_p:.1%}')
axes[0].set_title('N번 연속 음봉 후 다음 봉이 양봉일 확률')
axes[0].set_xlabel('연속 음봉 일수 (N)')
axes[0].set_ylabel('양봉 확률 (%)')
axes[0].legend()
for _, row in prob_df.iterrows():
    axes[0].text(row['n_reds'], row['p_next_green']*100 + 0.3,
                 f"{row['p_next_green']:.1%}", ha='center', fontsize=8)

# 평균 다음 봉 수익률
colors_ret = ['#2ecc71' if v > 0 else '#e74c3c' for v in prob_df['avg_next_ret_%']]
axes[1].bar(prob_df['n_reds'], prob_df['avg_next_ret_%'], color=colors_ret, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('N번 연속 음봉 후 다음 봉 평균 수익률')
axes[1].set_xlabel('연속 음봉 일수 (N)')
axes[1].set_ylabel('평균 다음 봉 수익률 (%)')
for _, row in prob_df.iterrows():
    axes[1].text(row['n_reds'], row['avg_next_ret_%'] + 0.02,
                 f"{row['avg_next_ret_%']:.3f}%", ha='center', fontsize=7)

# 발생 횟수
axes[2].bar(prob_df['n_reds'], prob_df['occurrences'], color='#3498db', edgecolor='white')
axes[2].set_title('N번 연속 음봉 발생 횟수')
axes[2].set_xlabel('연속 음봉 일수 (N)')
axes[2].set_ylabel('발생 횟수')
for _, row in prob_df.iterrows():
    axes[2].text(row['n_reds'], row['occurrences'] + 1,
                 str(int(row['occurrences'])), ha='center', fontsize=8)

plt.suptitle(f'{SYMBOL} 연속 음봉 후 다음 봉 분석 ({START}~{END})', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## 4. 더 깊이: N봉 후 1~5일 뒤 누적 수익률

다음 봉 하나만 보는 게 아니라, N일 후까지의 누적 성과를 본다.

In [ ]:
def forward_returns_after_n_reds(close, n_reds=3, max_forward=10):
    """N연속 음봉 후 1~max_forward일 뒤 평균 수익률."""
    ret = close.pct_change()
    is_red = (ret < 0).astype(int)
    trigger = is_red.rolling(n_reds).sum() == n_reds
    
    log_ret = np.log(close / close.shift(1))
    
    fwd_means = []
    fwd_stds  = []
    fwd_win   = []
    
    for d in range(1, max_forward + 1):
        cum_fwd = log_ret.rolling(d).sum().shift(-d)  # d일 누적 수익률 (선행)
        subset = cum_fwd[trigger].dropna()
        if len(subset) == 0:
            fwd_means.append(np.nan)
            fwd_stds.append(np.nan)
            fwd_win.append(np.nan)
        else:
            fwd_means.append(subset.mean() * 100)
            fwd_stds.append(subset.std() * 100)
            fwd_win.append((subset > 0).mean() * 100)
    
    return pd.DataFrame({
        'forward_days': range(1, max_forward + 1),
        'mean_ret_%': fwd_means,
        'std_%': fwd_stds,
        'win_rate_%': fwd_win,
    })

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors_n = ['#3498db', '#e67e22', '#e74c3c', '#9b59b6', '#1abc9c']
for i, n in enumerate([1, 2, 3, 4, 5]):
    fwd = forward_returns_after_n_reds(close, n_reds=n)
    c = colors_n[i]
    axes[0].plot(fwd['forward_days'], fwd['mean_ret_%'], marker='o', markersize=4,
                 label=f'{n}연속 음봉', color=c)
    axes[1].plot(fwd['forward_days'], fwd['win_rate_%'], marker='o', markersize=4,
                 label=f'{n}연속 음봉', color=c)
    axes[2].plot(fwd['forward_days'], fwd['mean_ret_%'] / fwd['std_%'], marker='o', markersize=4,
                 label=f'{n}연속 음봉', color=c)

axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('N연속 음봉 후 D일 평균 누적 수익률')
axes[0].set_xlabel('보유 일수')
axes[0].set_ylabel('평균 누적 수익률 (%)')
axes[0].legend(fontsize=8)

axes[1].axhline(50, color='black', linestyle='--', linewidth=0.8)
axes[1].set_title('N연속 음봉 후 D일 승률')
axes[1].set_xlabel('보유 일수')
axes[1].set_ylabel('승률 (%)')
axes[1].legend(fontsize=8)

axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].set_title('정보 비율 (평균수익/표준편차)')
axes[2].set_xlabel('보유 일수')
axes[2].set_ylabel('IR')
axes[2].legend(fontsize=8)

plt.suptitle(f'{SYMBOL} - N연속 음봉 후 D일 성과 ({START}~{END})', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## 5. 전략 백테스트: N번 음봉 후 매수, M일 보유

`signals.streak_signal()`로 실제 백테스트 실행.

In [ ]:
# 3번 연속 음봉 후 매수, 3일 보유
N_REDS   = 3
HOLD     = 3

entries, exits = signals.streak_signal(close, n_red=N_REDS, hold_days=HOLD, direction='red')

result = portfolio.from_signals(
    close=close,
    entries=entries,
    exits=exits,
    benchmark_close=benchmark,
    symbol=SYMBOL,
)

print(f'[{N_REDS}연속 음봉 후 매수, {HOLD}일 보유]')
print(result)
print()
display(tearsheet.metrics_table(result))

## 6. 파라미터 스윕: 몇 번 음봉 × 몇 일 보유가 최적인가?

In [ ]:
def streak_strategy(prices, n_red, hold_days):
    ent, ext = signals.streak_signal(prices, n_red=n_red, hold_days=hold_days, direction='red')
    return portfolio.from_signals(prices, ent, ext)

sweep_result = sweep.grid_search(
    strategy_fn=streak_strategy,
    prices=close,
    param_grid={
        'n_red':     [1, 2, 3, 4, 5, 6],
        'hold_days': [1, 2, 3, 5, 10],
    },
    metric='sharpe',
    extra_metrics=['cagr', 'max_dd', 'n_trades'],
)

print(f'총 {len(sweep_result.to_dataframe())} 조합 테스트')
print(f'안정성 지도: {sweep_result.stability_map("n_red", "hold_days"):.0%}')
print()
print('Top 10:')
display(sweep_result.best(n=10))

In [ ]:
sweep_result.heatmap('n_red', 'hold_days', metric='sharpe')

## 7. 통계적 유의성 검증

가장 좋아 보이는 파라미터 조합이 정말 유의한가?

In [ ]:
# 스윕에서 가장 좋은 조합으로 재백테스트
best_row = sweep_result.best(n=1).iloc[0]
best_n   = int(best_row['n_red'])
best_h   = int(best_row['hold_days'])
print(f'최적 파라미터: n_red={best_n}, hold_days={best_h}, sharpe={best_row["sharpe"]:.2f}')

ent_best, ext_best = signals.streak_signal(close, n_red=best_n, hold_days=best_h)
result_best = portfolio.from_signals(close, ent_best, ext_best, benchmark_close=benchmark)

strat_ret = result_best.returns

print()
print('=== t-test ===')
tt = stat_mod.ttest(strat_ret)
print(tt)

print()
print('=== Bootstrap ===')
bs = stat_mod.bootstrap_pvalue(strat_ret, n_simulations=5000)
print(bs)

In [ ]:
# Monte Carlo: 랜덤 진입 대비 몇 번째 백분위?
mc = stat_mod.monte_carlo(
    prices=close,
    n_simulations=1000,
    holding_days=best_h,
    observed_sharpe=result_best.metrics['sharpe'],
)
print(f'전략 Sharpe: {result_best.metrics["sharpe"]:.2f}')
print(f'랜덤 진입 대비 백분위: {mc.percentile():.1f}th')
print(f'p-value: {mc.p_value():.4f}')
mc.plot()

## 8. 롤링 백테스트: 시간에 따라 안정적인가?

In [ ]:
def streak_strategy_fixed(price_slice):
    ent, ext = signals.streak_signal(price_slice, n_red=N_REDS, hold_days=HOLD)
    return portfolio.from_signals(price_slice, ent, ext)

rr = rolling_bt.backtest(
    strategy_fn=streak_strategy_fixed,
    prices=close,
    window='4Y',
    step='6M',
    min_bars=60,
)

print(f'윈도우 수: {len(rr.windows)}')
print(f'안정성 스코어 (양의 Sharpe): {rr.stability_score():.0%}')
display(rr.summary())

rr.plot_metric('sharpe')

## 9. 레짐 분석: 강세장 vs 약세장에서 다른가?

직관적으로, 하락장에서 연속 음봉은 "반등"이 아니라 "추가 하락"으로 이어질 수 있다.

In [ ]:
strat_returns_main = result_best.returns
ra = regime_mod.RegimeAnalysis(returns=strat_returns_main, prices=close)

trend_rpt = ra.by_trend(ma_window=200)
print('=== MA200 강세/약세장 ===')
display(trend_rpt.summary())
trend_rpt.plot()

In [ ]:
try:
    vix_raw = yf.download('^VIX', start=START, end=END, auto_adjust=True, progress=False)
    vix = vix_raw['Close'].squeeze()
    vix.index = pd.to_datetime(vix.index).tz_localize('UTC')

    vix_rpt = ra.by_vix(vix)
    print('=== VIX 구간별 ===')
    display(vix_rpt.summary())
    vix_rpt.plot()
except Exception as e:
    print(f'VIX 없음: {e}')

## 10. 양봉도 똑같이 해보자: 연속 양봉 후 음봉 확률

In [ ]:
def prob_after_n_greens(close, max_n=8):
    """N번 연속 양봉 후 다음 봉의 방향 확률 분석."""
    ret = close.pct_change()
    is_green = (ret > 0).astype(int)
    is_red   = (ret < 0).astype(int)
    
    rows = []
    for n in range(1, max_n + 1):
        streak_n = is_green.rolling(n).sum() == n
        next_red = is_red.shift(-1)
        next_ret = ret.shift(-1)
        
        subset_red = next_red[streak_n].dropna()
        subset_ret = next_ret[streak_n].dropna()
        
        if len(subset_red) == 0:
            continue
        
        rows.append({
            'n_greens': n,
            'occurrences': len(subset_red),
            'p_next_red': subset_red.mean(),
            'avg_next_ret_%': subset_ret.mean() * 100,
        })
    return pd.DataFrame(rows)

green_df = prob_after_n_greens(close, max_n=9)
baseline_red = (close.pct_change() < 0).mean()
print(f'베이스라인 음봉 확률: {baseline_red:.1%}')
display(green_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 음봉 vs 양봉 이후 반전 확률 비교
n_vals_red = prob_df['n_reds'].values
p_green_after_red = prob_df['p_next_green'].values * 100

n_vals_green = green_df['n_greens'].values
p_red_after_green = green_df['p_next_red'].values * 100

axes[0].plot(n_vals_red, p_green_after_red, 'o-', color='#2ecc71', label='N연속 음봉 후 양봉 확률')
axes[0].plot(n_vals_green, p_red_after_green, 's--', color='#e74c3c', label='N연속 양봉 후 음봉 확률')
axes[0].axhline(50, color='black', linestyle=':', linewidth=1, label='50% (랜덤)')
axes[0].set_title('연속 캔들 후 반전 확률 비교')
axes[0].set_xlabel('연속 캔들 수 (N)')
axes[0].set_ylabel('반전 확률 (%)')
axes[0].legend(fontsize=9)
axes[0].set_ylim(30, 70)

# 평균 다음 봉 수익률 비교
axes[1].plot(n_vals_red, prob_df['avg_next_ret_%'].values, 'o-', color='#2ecc71', label='N연속 음봉 후')
axes[1].plot(n_vals_green, green_df['avg_next_ret_%'].values, 's--', color='#e74c3c', label='N연속 양봉 후')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('연속 캔들 후 다음 봉 평균 수익률')
axes[1].set_xlabel('연속 캔들 수 (N)')
axes[1].set_ylabel('평균 수익률 (%)')
axes[1].legend(fontsize=9)

plt.suptitle(f'{SYMBOL} 연속 캔들 패턴 요약 ({START}~{END})', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## 11. 종합 Tearsheet

In [ ]:
tearsheet.plot(
    result_best,
    benchmark_close=benchmark,
    title=f'연속 음봉 반등 전략 (n={best_n}, hold={best_h}d) - {SYMBOL}',
)

## 결론 체크리스트

| 분석 항목 | 확인 방법 | 결과 |
|---|---|---|
| 음봉 후 양봉 확률 > 50%? | Section 3 bar chart | ??? |
| 음봉 길수록 반등 강해짐? | Section 3 trend | ??? |
| 백테스트 수익성? | Section 5 metrics | ??? |
| 통계적 유의성 (p < 0.05)? | Section 7 | ??? |
| 안정성 (롤링 Sharpe 분포)? | Section 8 | ??? |
| 강세장/약세장 차이? | Section 9 | ??? |
| 과최적화 위험 (파라미터 민감도)? | Section 6 heatmap | ??? |

**해석 가이드:**
- 확률 50% 이상이어도, 수익률 기댓값이 작으면 실제 전략으로 쓰기 어렵다.
- 강세장에서만 효과가 있다면, MA200 필터를 추가하는 것을 고려하라.
- 파라미터 히트맵이 고르게 양수라면 robust한 전략. 특정 값에서만 좋다면 과최적화.
